# Normalize Article Text

This notebook prepares Khmer article text for search. It applies Unicode and whitespace normalization only. It does not translate, correct legal wording, tokenize, segment, or generate embeddings.

In [ ]:
import json
import re
import unicodedata
from pathlib import Path

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'react-client').is_dir() and (path / 'data-pipeline').is_dir()
)
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
INPUT_PATH = DATA_DIR / 'processed' / 'articles.jsonl'
OUTPUT_PATH = DATA_DIR / 'processed' / 'articles_normalized.jsonl'
ZERO_WIDTH = '\u200b\u200c\u200d\ufeff'
SPELLING_VARIANTS = {
    'សំលៀកបំពាក់': 'សម្លៀកបំពាក់',
    'ចំរៀង': 'ចម្រៀង',
    'ជ័យជម្នះ': 'ជ័យជំនះ',
}
print('Input:', INPUT_PATH)
print('Output:', OUTPUT_PATH)

In [ ]:
def normalize_khmer_cluster_order(text):
    # Canonically order combining marks without removing Khmer characters.
    decomposed = unicodedata.normalize('NFD', text)
    clusters, current = [], ''
    for char in decomposed:
        if unicodedata.combining(char) == 0 and current:
            clusters.append(current)
            current = ''
        current += char
    if current:
        clusters.append(current)
    ordered = []
    for cluster in clusters:
        if len(cluster) > 1:
            base, marks = cluster[0], list(cluster[1:])
            marks.sort(key=unicodedata.combining)
            cluster = base + ''.join(marks)
        ordered.append(cluster)
    return unicodedata.normalize('NFC', ''.join(ordered))

def normalize_khmer(value):
    text = normalize_khmer_cluster_order(str(value or ''))
    text = text.translate(str.maketrans('', '', ZERO_WIDTH))
    text = text.replace('\u00a0', ' ')
    lines = []
    for line in text.splitlines():
        line = re.sub(r'[ \t]+', ' ', line).strip()
        if line:
            lines.append(line)
    text = re.sub(r'\n{2,}', '\n', '\n'.join(lines))
    for source, target in SPELLING_VARIANTS.items():
        text = text.replace(source, target)
    return text


In [ ]:
records = []
for line in INPUT_PATH.read_text(encoding='utf-8').splitlines():
    record = json.loads(line)
    normalized = dict(record)
    normalized['search_text'] = normalize_khmer(record['text'])
    normalized['normalization_version'] = 'unicode-khmer-cluster-order-spelling-v2'
    records.append(normalized)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open('w', encoding='utf-8') as output:
    for record in records:
        output.write(json.dumps(record, ensure_ascii=False) + '\n')

print(f'Normalized articles: {len(records):,}')
print(f'Output: {OUTPUT_PATH}')

In [ ]:
assert records
assert len(records) == len({record['id'] for record in records})
assert all(record['search_text'].strip() for record in records)
assert not any(any(character in record['search_text'] for character in ZERO_WIDTH) for record in records)
print('Validation passed')
print(json.dumps(records[0], ensure_ascii=False, indent=2))